In [1]:
# === SESSION BOOTSTRAP (run first in every notebook) ======================
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT = "/content/drive/MyDrive/UAV_TRUST_Research"
REPO   = f"{PARENT}/uav-trust-research"
for fname in (".gitconfig", ".git-credentials"):
    src = os.path.join(PARENT, fname)
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{fname}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
r = subprocess.run("git config --global user.name && git config --global user.email",
                   shell=True, capture_output=True, text=True)
print("git identity:", r.stdout.strip() or "MISSING - run 00_setup.ipynb first")
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0, REPO) if REPO not in sys.path else None
    print("cwd:", os.getcwd())
else:
    print("Repository not on Drive yet - run 00_setup.ipynb first.")

Mounted at /content/drive
git identity: Md Anas Biswas
anasbiswas@gmail.com
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost shap scikit-learn matplotlib pandas numpy scipy pyarrow requests --quiet

In [3]:
# Two dataset specs; the completion analysis runs on both.
DATASETS = [
  {"name": "UAVIDS-2025", "kind": "zenodo", "record": "15336998",
   "data_dir": "data/uavids2025", "label_col": "label", "normal_value": "Normal Traffic",
   "include_families": None, "subsample_n": None,
   "drops": ["unnamed","flowid","srcaddr","dstaddr","srcport","dstport","index","timestamp"]},
  {"name": "UAV-NIDD", "kind": "file",
   "file": "data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv", "parquet": "data/uav_nidd/case1.parquet",
   "data_dir": "data/uav_nidd", "label_col": "Label", "normal_value": "Normal",
   "include_families": ["DDoS","UDP Flooding","MITM","Jamming","BruteForce","De-authentication"],
   "subsample_n": None,
   "drops": ["unnamed","index","ip.src","ip.dst","ip.proto","srcport","dstport","udp.srcport","udp.dstport",
             "frame.time","frame.number","time_epoch","time_relative","time_delta","bssid","mactime",
             "vendor_oui","wlan_radio.timestamp","wlan_radio.start_tsf","radiotap.timestamp","wlan.seq"]},
]
CFG = {"seeds": list(range(10)), "alpha": 0.10, "nbins": 15, "n_shap": 2000,
       "normal_fracs": {"train":0.60,"cal":0.20,"test_seen":0.10,"test_shift":0.10},
       "family_fracs": {"train":0.60,"cal":0.20,"test_seen":0.20},
       "xgb": {"n_estimators":300,"max_depth":6,"learning_rate":0.1,"subsample":0.9,
               "colsample_bytree":0.9,"tree_method":"hist"},
       "fig_dir":"figures", "report_dir":"reports"}
for d in [CFG["fig_dir"], CFG["report_dir"]]: os.makedirs(d, exist_ok=True)
print("configured")

configured


In [4]:
import numpy as np, pandas as pd, requests, glob, zipfile
import matplotlib.pyplot as plt
import xgboost as xgb, shap
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from src.trust import (top_label_ece, brier_binary, conformal_qhat, coverage_size,
                       aurc, logit, fit_calibrators, apply_calibrators)
from src.data import load_csvs, detect_schema, prepare_splits

def load_dataset(spec):
    dd = spec["data_dir"]; os.makedirs(dd, exist_ok=True)
    if spec["kind"] == "zenodo":
        if not glob.glob(dd+"/**/*.csv", recursive=True):
            meta = requests.get(f"https://zenodo.org/api/records/{spec['record']}", timeout=60).json()
            for f in meta.get("files", []):
                n,u = f["key"], f["links"]["self"]
                if n.lower().endswith((".csv",".zip",".gz")):
                    open(os.path.join(dd,n),"wb").write(requests.get(u,timeout=1200).content)
            for z in glob.glob(dd+"/*.zip"): zipfile.ZipFile(z).extractall(dd)
        df = load_csvs(dd); lc, nv, fams = detect_schema(df, spec["label_col"], spec["normal_value"])
    else:
        pq = spec.get("parquet")
        if pq and os.path.exists(pq): df = pd.read_parquet(pq)
        else:
            df = pd.read_csv(spec["file"], low_memory=False, encoding="latin-1")
            if pq:
                try: df.to_parquet(pq)
                except Exception as e: print("parquet skip:", e)
        lc, nv = spec["label_col"], spec["normal_value"]
        fams = [v for v in df[lc].unique() if v != nv]
    if spec.get("subsample_n") and len(df) > spec["subsample_n"]:
        frac = spec["subsample_n"]/len(df)
        df = df.groupby(lc, group_keys=False).sample(frac=frac, random_state=42).reset_index(drop=True)
    if spec.get("include_families"):
        df = df[df[lc].isin([nv]+list(spec["include_families"]))].reset_index(drop=True)
        fams = list(spec["include_families"])
    return df, lc, nv, fams

def coverage_detail(p, y, qhat):
    p2 = np.column_stack([1-np.asarray(p), np.asarray(p)])
    sets = p2 >= (1 - qhat); y = np.asarray(y)
    inset = sets[np.arange(len(y)), y]
    marg = float(inset.mean())
    bal = float(np.mean([inset[y==k].mean() for k in np.unique(y)]))
    return marg, bal, float(sets.sum(1).mean())
print("helpers ready")

helpers ready


In [5]:
# Full trust panel per (family, seed) with SPLIT calibration:
# model fixed on train; probability calibrators fit on prob-cal half;
# conformal quantile from raw scores on a DISJOINT conf-cal half.
def panel_once(df, lc, nv, F, seed, spec):
    S = prepare_splits(df, lc, nv, F, spec["drops"], CFG["normal_fracs"], CFG["family_fracs"], seed)
    clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=seed, **CFG["xgb"]).fit(S["X_train"], S["y_train"])
    Xc, yc = S["X_cal"], S["y_cal"]
    perm = np.random.default_rng(seed).permutation(len(yc)); h = len(yc)//2
    ip, ic = perm[:h], perm[h:]                      # disjoint prob-cal / conf-cal
    prob = lambda X: (clf.predict_proba(X)[:,1], logit(clf.predict_proba(X)[:,1]))
    p_pc, lg_pc = prob(Xc[ip]); p_seen, lg_seen = prob(S["X_seen"]); p_shift, lg_shift = prob(S["X_shift"])
    p_cc = clf.predict_proba(Xc[ic])[:,1]
    y_seen, y_shift = S["y_seen"], S["y_shift"]
    fitted = fit_calibrators(lg_pc, p_pc, yc[ip])                  # calibrators on prob-cal only
    cs = apply_calibrators(fitted, lg_seen, p_seen); ch = apply_calibrators(fitted, lg_shift, p_shift)
    qhat = conformal_qhat(p_cc, yc[ic], alpha=CFG["alpha"])        # quantile on conf-cal, RAW scores
    cm_h, cb_h, sz_h = coverage_detail(p_shift, y_shift, qhat)
    cm_s, cb_s, sz_s = coverage_detail(p_seen, y_seen, qhat)
    pr_h = (p_shift>=0.5).astype(int); pr_s = (p_seen>=0.5).astype(int)
    row = {"dataset": spec["name"], "held_out": str(F), "seed": seed,
           "shift_balacc": balanced_accuracy_score(y_shift, pr_h),
           "shift_Brier_temp": brier_binary(ch["temperature"], y_shift),
           "marginal_cov_shift": cm_h, "balanced_cov_shift": cb_h, "setsize_shift": sz_h,
           "marginal_cov_seen": cm_s, "balanced_cov_seen": cb_s,
           "shift_AURC": aurc(np.maximum(p_shift,1-p_shift),(pr_h==y_shift).astype(float))[0]}
    for cal in ["raw","temperature","platt","isotonic"]:      # calibrator comparison, seen vs shift
        row[f"ECE_{cal}_seen"]  = top_label_ece(cs[cal], y_seen, CFG["nbins"])
        row[f"ECE_{cal}_shift"] = top_label_ece(ch[cal], y_shift, CFG["nbins"])
        row[f"Brier_{cal}_shift"] = brier_binary(ch[cal], y_shift)
    return row
print("panel_once defined")

panel_once defined


In [6]:
# Run the panel on both datasets across seeds
panel_rows = []
loaded = {}
for spec in DATASETS:
    df, lc, nv, fams = load_dataset(spec); loaded[spec["name"]] = (df, lc, nv, fams)
    for seed in CFG["seeds"]:
        for F in fams:
            panel_rows.append(panel_once(df, lc, nv, F, seed, spec))
        print(spec["name"], "seed", seed, "done")
panel = pd.DataFrame(panel_rows)
panel.to_csv(os.path.join(CFG["report_dir"], "08_full_panel_raw.csv"), index=False)
print("panel rows:", len(panel))

UAVIDS-2025 seed 0 done
UAVIDS-2025 seed 1 done
UAVIDS-2025 seed 2 done
UAVIDS-2025 seed 3 done
UAVIDS-2025 seed 4 done
UAVIDS-2025 seed 5 done
UAVIDS-2025 seed 6 done
UAVIDS-2025 seed 7 done
UAVIDS-2025 seed 8 done
UAVIDS-2025 seed 9 done
UAV-NIDD seed 0 done
UAV-NIDD seed 1 done
UAV-NIDD seed 2 done
UAV-NIDD seed 3 done
UAV-NIDD seed 4 done
UAV-NIDD seed 5 done
UAV-NIDD seed 6 done
UAV-NIDD seed 7 done
UAV-NIDD seed 8 done
UAV-NIDD seed 9 done
panel rows: 100


In [7]:
import pandas as pd, os
print(os.path.exists("reports/08_full_panel_raw.csv"))
panel = pd.read_csv("reports/08_full_panel_raw.csv")
print(panel.shape)

True
(100, 23)


In [8]:
# TABLE: completed trust panel per family (means), both datasets
cols = ["shift_balacc","ECE_temperature_shift","shift_Brier_temp",
        "marginal_cov_shift","balanced_cov_shift","setsize_shift","shift_AURC"]
tab = panel.groupby(["dataset","held_out"])[cols].mean().round(4)
tab.columns = ["bal_acc","ECE","Brier","cov_marginal","cov_balanced","set_size","AURC"]
print(tab.to_string())
tab.to_csv(os.path.join(CFG["report_dir"], "08_trust_panel_by_family.csv"))

                               bal_acc     ECE   Brier  cov_marginal  cov_balanced  set_size    AURC
dataset     held_out                                                                                
UAV-NIDD    BruteForce          0.9984  0.0007  0.0008        0.6317        0.3747    0.6317  0.0000
            DDoS                0.9742  0.0422  0.0346        0.0262        0.0648    0.0262  0.0043
            De-authentication   0.9989  0.0003  0.0002        0.7900        0.4611    0.7900  0.0000
            Jamming             0.9720  0.0519  0.0444        0.0060        0.0730    0.0060  0.0083
            MITM                0.9985  0.0008  0.0006        0.3846        0.2500    0.3846  0.0002
            UDP Flooding        0.9988  0.0002  0.0000        0.8268        0.5080    0.8268  0.0000
UAVIDS-2025 Blackhole Attack    0.7306  0.4050  0.3903        0.4083        0.5989    0.4083  0.1896
            Flooding Attack     0.9929  0.0081  0.0053        0.7176        0.7614    0.717

In [9]:
# TABLE: recalibration does not fix shift -- all calibrators, seen vs shift (averaged)
rows = []
for cal in ["raw","temperature","platt","isotonic"]:
    rows.append({"calibrator": cal,
                 "ECE_seen":  panel[f"ECE_{cal}_seen"].mean(),
                 "ECE_shift": panel[f"ECE_{cal}_shift"].mean(),
                 "Brier_shift": panel[f"Brier_{cal}_shift"].mean()})
caltab = pd.DataFrame(rows).round(4)
print(caltab.to_string(index=False))
caltab.to_csv(os.path.join(CFG["report_dir"], "08_calibrator_comparison.csv"), index=False)

 calibrator  ECE_seen  ECE_shift  Brier_shift
        raw    0.0004     0.0651       0.0629
temperature    0.0004     0.0667       0.0637
      platt    0.0004     0.0651       0.0632
   isotonic    0.0006     0.0733       0.0677


In [10]:
# MARGINAL vs BALANCED coverage side by side (point 5)
cov = panel.groupby(["dataset","held_out"])[["marginal_cov_shift","balanced_cov_shift"]].mean().round(4)
cov.columns = ["coverage_marginal","coverage_balanced"]
print(cov.to_string())
cov.to_csv(os.path.join(CFG["report_dir"], "08_marginal_vs_balanced_coverage.csv"))

                               coverage_marginal  coverage_balanced
dataset     held_out                                               
UAV-NIDD    BruteForce                    0.6317             0.3747
            DDoS                          0.0262             0.0648
            De-authentication             0.7900             0.4611
            Jamming                       0.0060             0.0730
            MITM                          0.3846             0.2500
            UDP Flooding                  0.8268             0.5080
UAVIDS-2025 Blackhole Attack              0.4083             0.5989
            Flooding Attack               0.7176             0.7614
            Sybil Attack                  0.9116             0.8743
            Wormhole Attack               0.6350             0.7201


In [11]:
loaded = {}
for spec in DATASETS:
    df, lc, nv, fams = load_dataset(spec)
    loaded[spec["name"]] = (df, lc, nv, fams)
    print(spec["name"], "loaded:", df.shape)

UAVIDS-2025 loaded: (122171, 23)
UAV-NIDD loaded: (747621, 45)


In [12]:
# NORMALIZED fragility (point 6): raw score / total attack-support mass, both datasets, seed 0
def mean_shap(expl, Xg):
    s = np.asarray(expl.shap_values(Xg))
    if s.ndim == 3: s = s[..., 1]
    return s.mean(0)
def samp(A, n, seed):
    if len(A) > n:
        j = np.random.default_rng(seed).choice(len(A), n, replace=False); return A[j]
    return A

frag_rows, fragstore = [], {}
for spec in DATASETS:
    df, lc, nv, fams = loaded[spec["name"]]
    for F in fams:
        S = prepare_splits(df, lc, nv, F, spec["drops"], CFG["normal_fracs"], CFG["family_fracs"], 0)
        clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                random_state=0, **CFG["xgb"]).fit(S["X_train"], S["y_train"])
        Xa = samp(S["X_seen"][np.isin(S["fam_seen"], S["seen_families"])], CFG["n_shap"], 0)
        Xh = samp(S["X_shift"][S["fam_shift"]==F], CFG["n_shap"], 0)
        ex = shap.TreeExplainer(clf); m_s, m_h = mean_shap(ex, Xa), mean_shap(ex, Xh)
        ap = np.maximum(m_s,0); flip = ap*np.maximum(-m_h,0)
        raw = float(flip.sum()); norm = float(flip.sum()/(ap.sum()+1e-12))
        frag_rows.append({"dataset": spec["name"], "held_out": str(F),
                          "fragility_raw": round(raw,4), "fragility_norm": round(norm,4)})
        fragstore[(spec["name"], str(F))] = dict(feats=S["feature_names"], flip=flip, S=S)
    print(spec["name"], "fragility done")
frag = pd.DataFrame(frag_rows)
print(frag.to_string(index=False))
frag.to_csv(os.path.join(CFG["report_dir"], "08_fragility_normalized.csv"), index=False)

UAVIDS-2025 fragility done
UAV-NIDD fragility done
    dataset          held_out  fragility_raw  fragility_norm
UAVIDS-2025      Sybil Attack         2.5859          0.2544
UAVIDS-2025  Blackhole Attack         4.9801          0.4806
UAVIDS-2025   Wormhole Attack         3.0317          0.2892
UAVIDS-2025   Flooding Attack         1.6519          0.1573
   UAV-NIDD              DDoS         6.5060          0.7066
   UAV-NIDD      UDP Flooding         0.0000          0.0000
   UAV-NIDD              MITM         0.9948          0.1138
   UAV-NIDD           Jamming         7.5600          0.9042
   UAV-NIDD        BruteForce         1.0599          0.1179
   UAV-NIDD De-authentication         2.0214          0.2404


In [13]:
# VALIDATION by causal ablation (point 6): remove the top transfer-fragile features from the
# worst family in each dataset, retrain, and check whether shift coverage/accuracy improve.
def evaluate(df, lc, nv, F, spec, seed=0, drop_feat_idx=None):
    S = prepare_splits(df, lc, nv, F, spec["drops"], CFG["normal_fracs"], CFG["family_fracs"], seed)
    keep = np.arange(S["X_train"].shape[1])
    if drop_feat_idx is not None:
        keep = np.array([i for i in keep if i not in set(drop_feat_idx)])
    clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=seed, **CFG["xgb"]).fit(S["X_train"][:,keep], S["y_train"])
    p_cal = clf.predict_proba(S["X_cal"][:,keep])[:,1]
    p_shift = clf.predict_proba(S["X_shift"][:,keep])[:,1]
    qhat = conformal_qhat(p_cal, S["y_cal"], alpha=CFG["alpha"])
    _, cb, _ = coverage_detail(p_shift, S["y_shift"], qhat)
    ba = balanced_accuracy_score(S["y_shift"], (p_shift>=0.5).astype(int))
    return ba, cb

ablation = []
for spec in DATASETS:
    df, lc, nv, fams = loaded[spec["name"]]
    worst = frag[frag.dataset==spec["name"]].sort_values("fragility_raw").iloc[-1]["held_out"]
    flip = fragstore[(spec["name"], worst)]["flip"]
    topk = list(np.argsort(-flip)[:3])                          # top-3 transfer-fragile features
    ba0, cb0 = evaluate(df, lc, nv, worst, spec, drop_feat_idx=None)
    ba1, cb1 = evaluate(df, lc, nv, worst, spec, drop_feat_idx=topk)
    ablation.append({"dataset": spec["name"], "worst_family": worst,
                     "balacc_full": round(ba0,4), "balacc_ablated": round(ba1,4),
                     "balcov_full": round(cb0,4), "balcov_ablated": round(cb1,4)})
    print(spec["name"], worst, "coverage %.3f -> %.3f (ablated top-fragile)" % (cb0, cb1))
abl = pd.DataFrame(ablation)
print(abl.to_string(index=False))
abl.to_csv(os.path.join(CFG["report_dir"], "08_fragility_ablation.csv"), index=False)

UAVIDS-2025 Blackhole Attack coverage 0.588 -> 0.624 (ablated top-fragile)
UAV-NIDD Jamming coverage 0.000 -> 0.504 (ablated top-fragile)
    dataset     worst_family  balacc_full  balacc_ablated  balcov_full  balcov_ablated
UAVIDS-2025 Blackhole Attack       0.7338          0.8552       0.5882          0.6236
   UAV-NIDD          Jamming       0.9798          0.9891       0.0000          0.5044


In [ ]:
# FIGURE: risk-coverage curves for the worst family in each dataset (seed 0)
fig, ax = plt.subplots(1, len(DATASETS), figsize=(6*len(DATASETS), 4.2))
ax = np.atleast_1d(ax)
for a, spec in zip(ax, DATASETS):
    df, lc, nv, fams = loaded[spec["name"]]
    worst = frag[frag.dataset==spec["name"]].sort_values("fragility_raw").iloc[-1]["held_out"]
    S = prepare_splits(df, lc, nv, worst, spec["drops"], CFG["normal_fracs"], CFG["family_fracs"], 0)
    clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=0, **CFG["xgb"]).fit(S["X_train"], S["y_train"])
    for tag, X, y, col in [("seen test", S["X_seen"], S["y_seen"], "#2e7d32"),
                            ("shift ("+worst[:10]+")", S["X_shift"], S["y_shift"], "#c62828")]:
        p = clf.predict_proba(X)[:,1]; conf = np.maximum(p,1-p)
        correct = ((p>=0.5).astype(int)==np.asarray(y)).astype(float)
        _, covs, risk = aurc(conf, correct)
        a.plot(covs, risk, label=tag, color=col)
    a.set_xlabel("coverage"); a.set_ylabel("selective risk"); a.set_title(spec["name"]); a.legend(fontsize=8)
fig.suptitle("Risk-coverage curves: worst-transferring family, seen vs shift")
fig.tight_layout(); fig.savefig(os.path.join(CFG["fig_dir"], "08_risk_coverage.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Commit
!git add reports/ figures/ notebooks/
!git commit -m "08 trust-panel completion: split calibration, Brier/calibrators/set-size/AURC/risk-coverage, marginal vs balanced coverage, normalized+ablation-validated fragility (both datasets)"
!git push origin main